# Figure 3 MAPS runtime remeasurement for all 34 datasets

This notebook runs the 29 common datasets followed by `P19`, `IW`, `TA`, `GSS`, and `PAM` on one Colab L4. Exactly one dataset process runs at a time. Every dataset uses seeds 2026--2028; `PAM` uses batch size 16 and all other datasets use batch size 32. Results are isolated in Google Drive and successful rows are skipped when the coordinator is restarted. The notebook assumes all anchor sweep CSVs are available under `pypots-pyrregular/testing_results/mascit_train_loss_only_single_dataset/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone/update the runner repository and initialize its PyPOTS submodule.
from google.colab import userdata
from pathlib import Path

# token = userdata.get('GH_TOKEN')
# if not token:
#     raise RuntimeError('GH_TOKEN is not set in Colab secrets')

runner_repo = Path('/content/pypots-pyrregular')
if runner_repo.exists():
    %cd /content/pypots-pyrregular
    !git fetch origin main
    !git checkout main
    !git pull --ff-only origin main
    !git submodule update --init --recursive
else:
    # !git clone --recurse-submodules https://{token}@github.com/yoom618/pypots-pyrregular.git /content/pypots-pyrregular
    !git clone --recurse-submodules https://github.com/yoom618/pypots-pyrregular.git /content/pypots-pyrregular
%cd /content/pypots-pyrregular
!git status --short --branch
!git rev-parse HEAD

In [ ]:
# Verify that the assigned runtime is an L4.
import sys, torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A GPU runtime is required')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
REQUIRE_L4 = True
if REQUIRE_L4 and 'L4' not in gpu_name.upper():
    raise RuntimeError(f'Expected an L4, but Colab assigned: {gpu_name}')

In [ ]:
# Install PyPOTS dependencies first, then the editable submodule and Mamba extensions.
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r /content/pypots-pyrregular/requirements.txt
!python -m pip install -q -e /content/pypots-pyrregular/PyPOTS
%pip install -q causal-conv1d --no-build-isolation
%pip install -q mamba-ssm --no-build-isolation

In [ ]:
# Runtime paths. Results persist in Drive; downloaded datasets remain in Colab cache.
from pathlib import Path
import os

PROJECT_ROOT = Path('/content/pypots-pyrregular')
DATASET_DIR = Path('/content/dataset')
ANCHOR_DIR = PROJECT_ROOT / 'testing_results' / 'mascit_train_loss_only_single_dataset'
OUTPUT_ROOT = Path('/content/drive/MyDrive/pyrregular_testing_results/figure3_l4_runtime_all34')
OUTPUT_DIR = OUTPUT_ROOT / 'results'
LOG_PATH = OUTPUT_ROOT / 'figure3_l4_runtime_all34.log'
PID_PATH = OUTPUT_ROOT / 'figure3_l4_runtime_all34.pid'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + str(PROJECT_ROOT / 'PyPOTS') + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Dataset cache:', DATASET_DIR)
print('Anchor CSV directory:', ANCHOR_DIR)
print('Result CSV directory:', OUTPUT_DIR)
print('Log:', LOG_PATH)

In [ ]:
# Freeze the complete execution order and validate all 34 anchors before training.
COMMON_29 = ['ABF','AN','AOC','APT','ARC','CT','DD','DG','DW',
             'GM1','GM2','GM3','GP1','GP2','GX','GY','GZ',
             'JV','LPA','MI3','MP','P12','PGE','PGZ',
             'PL','SAD','SE','SGZ','VE']
ADDITIONAL_5 = ['P19','IW','TA','GSS','PAM'] # not runned in some baselines.
ALL_34 = COMMON_29 + ADDITIONAL_5
assert len(ALL_34) == 34 and len(set(ALL_34)) == 34
missing = [dataset for dataset in ALL_34 if not (ANCHOR_DIR / f'{dataset}_sweep_results.csv').exists()]
if missing:
    raise RuntimeError(f'Missing anchor sweep CSVs: {missing}')
print('Execution order:', ALL_34)
for dataset in ALL_34:
    !python scripts/run_figure3_l4_runtime_single_dataset.py --dataset {dataset} --dataset-dir {DATASET_DIR} --anchor-dir {ANCHOR_DIR} --dry-run

## Launch all 34 datasets sequentially

The background coordinator starts a dataset only after the preceding Python process exits. Re-running this cell resumes from the existing result CSVs because successful seed/configuration rows are skipped.

In [ ]:
# COMMON_29 -> P19 -> IW -> TA -> GSS -> PAM.
!nohup bash -lc 'set -euo pipefail; cd /content/pypots-pyrregular; datasets=(ABF AN AOC APT ARC CT DD DG DW GM1 GM2 GM3 GP1 GP2 GX GY GZ JV LPA MI3 MP P12 PGE PGZ PL SAD SE SGZ VE P19 IW TA GSS PAM); for dataset in "${datasets[@]}"; do batch_size=32; if [ "$dataset" = PAM ]; then batch_size=16; fi; echo "===== START ${dataset} batch=${batch_size} $(date --iso-8601=seconds) ====="; if python scripts/run_figure3_l4_runtime_single_dataset.py --dataset "$dataset" --dataset-dir /content/dataset --anchor-dir /content/pypots-pyrregular/testing_results/mascit_train_loss_only_single_dataset --output-dir /content/drive/MyDrive/pyrregular_testing_results/figure3_l4_runtime_all34/results --device cuda --batch-size "$batch_size" --epochs 100 --patience 10; then echo "SUCCESS ${dataset}"; else echo "FAILED ${dataset}"; fi; echo "===== END ${dataset} $(date --iso-8601=seconds) ====="; done' > /content/drive/MyDrive/pyrregular_testing_results/figure3_l4_runtime_all34/figure3_l4_runtime_all34.log 2>&1 & echo $! > /content/drive/MyDrive/pyrregular_testing_results/figure3_l4_runtime_all34/figure3_l4_runtime_all34.pid
!cat /content/drive/MyDrive/pyrregular_testing_results/figure3_l4_runtime_all34/figure3_l4_runtime_all34.pid

In [ ]:
# Progress snapshot. Final coverage is 102 successful rows across 34 datasets.
import pandas as pd

files = sorted(OUTPUT_DIR.glob('*_l4_runtime_results.csv'))
frames = []
for path in files:
    frame = pd.read_csv(path)
    frame['source_file'] = path.name
    frames.append(frame)
if frames:
    progress = pd.concat(frames, ignore_index=True)
    ok = progress['error'].fillna('').eq('') & progress['f1_macro'].notna()
    successful = progress.loc[ok].drop_duplicates(['dataset_abbr','seed','cfg_hash'], keep='last')
    complete = successful.groupby('dataset_abbr').seed.nunique().eq(3)
    print(f'Files: {len(files)}/34')
    print(f'Successful rows: {len(successful)}/102')
    print(f'Datasets with 3 successful seeds: {int(complete.sum())}/34')
    display(successful[['dataset_abbr','seed','tv_pattern','batch_size','total_runtime_seconds','f1_macro','error']].tail(15))
else:
    print('No result CSV has been written yet.')
!tail -n 30 /content/drive/MyDrive/pyrregular_testing_results/figure3_l4_runtime_all34/figure3_l4_runtime_all34.log

In [ ]:
# Build a combined CSV only after all 102 expected rows are complete.
import pandas as pd

files = sorted(OUTPUT_DIR.glob('*_l4_runtime_results.csv'))
if not files:
    raise RuntimeError('No result CSVs found')
combined = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
ok = combined['error'].fillna('').eq('') & combined['f1_macro'].notna()
combined = combined.loc[ok].drop_duplicates(['dataset_abbr','seed','cfg_hash'], keep='last')
order = {dataset: index for index, dataset in enumerate(ALL_34)}
combined['dataset_order'] = combined['dataset_abbr'].map(order)
combined = combined.sort_values(['dataset_order','seed']).drop(columns='dataset_order')
if len(combined) != 102 or combined['dataset_abbr'].nunique() != 34:
    raise RuntimeError(f'Incomplete result: {len(combined)} successful rows across {combined.dataset_abbr.nunique()} datasets')
expected_batch = {dataset: (16 if dataset == 'PAM' else 32) for dataset in ALL_34}
actual_batch = combined.groupby('dataset_abbr')['batch_size'].unique().apply(list).to_dict()
if any(actual_batch.get(dataset) != [batch] for dataset, batch in expected_batch.items()):
    raise RuntimeError(f'Unexpected batch sizes: {actual_batch}')
combined_path = OUTPUT_DIR / 'figure3_l4_runtime_all34.csv'
combined.to_csv(combined_path, index=False)
print('Saved:', combined_path)
display(combined.groupby('dataset_abbr', sort=False).agg(runtime_mean=('total_runtime_seconds','mean'), f1_mean=('f1_macro','mean'), batch_size=('batch_size','first')))

In [ ]:
# Optional emergency stop. Do not run during normal execution.
# pid = int(PID_PATH.read_text().strip())
# os.kill(pid, 15)
# print('Sent SIGTERM to coordinator:', pid)